In [7]:
import os, json, re, time
import pandas as pd
from openai import OpenAI


In [8]:
client = OpenAI(
    api_key="AQ.Ab8RN6Key-eQoaYXkDO-0wZkyKF_I7Z8SkfJcmvn67kREoIjRw",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [9]:
QUESTIONS = [
    "1.您认为该思维片段多大可能是【抑郁症患者】说的。[1:完全不是; 9:绝对是]",
    "2.您认为该思维片段多大程度上是反映说话者【内部感受】的（躯体，内心感觉）。[1:完全没有; 9:几乎都是]",
    "3.您认为该思维片段多大程度上是【内部导向】的（独立于外部刺激，自发产生的想法/画面）。[1:完全没有; 9:几乎都是]",
    "4.您认为该思维片段多大程度上是【外部导向】的（由外界刺激诱发的想法，例如噪声、设备、实验任务等）。[1:完全没有; 9:几乎都是]",
    "5.您认为该思维片段多大程度上是关注【自我】的。[1:完全没有; 9:几乎都是]",
    "6.您认为该思维片段多大程度上是关注【他人】的。[1:完全没有; 9:几乎都是]",
    "7.您认为该思维片段多大程度上说的是【正性的、积极】的事情。[1:完全没有; 9:几乎都是]",
    "8.您认为该思维片段多大程度上说的是【负性的、消极】的事情。[1:完全没有; 9:几乎都是]",
    "9.您认为该思维片段涉及【问题解决】的程度。（试图解决某个问题，或思考如何达到某个目标）。[1:完全没有; 9:几乎都是]",
    "10.您认为该思维片段反映说话者【自责、内疚】的程度。[1:完全没有; 9:几乎都是]",
    "11.您认为该思维片段反映说话者【担忧】的程度。[1:完全没有; 9:几乎都是]",
    "12.您认为该思维片段反映说话者【生气】的程度。[1:完全没有; 9:几乎都是]",
    "13.您认为该思维片段反映说话者【厌恶】的程度。[1:完全没有; 9:几乎都是]",
    "14.您认为该思维片段反映说话者【恐惧】的程度。[1:完全没有; 9:几乎都是]",
    "15.您认为该思维片段反映说话者【悲伤】的程度。[1:完全没有; 9:几乎都是]",
    "16.您认为该思维片段反映说话者【惊喜】的程度。[1:完全没有; 9:几乎都是]",
    "17.您认为该思维片段反映说话者【幸福】的程度。[1:完全没有; 9:几乎都是]",
    "18.您认为该思维片段反映说话者【喜欢】的程度。[1:完全没有; 9:几乎都是]",
    "19.您认为该思维片段反映说话者【愉悦】的程度。[1:完全没有; 9:几乎都是]",
    "20.您认为该思维片段反映说话者【自卑】的程度。[1:完全没有; 9:几乎都是]",
    "21.您认为该思维片段反映说话者【失望】的程度。[1:完全没有; 9:几乎都是]",
    "22.您认为该思维片段反映说话者【反省】的程度。[1:完全没有; 9:几乎都是]",
    "23.您认为该思维片段多大程度上是关注【过去】的事情。[1:完全没有; 9:几乎都是]",
    "24.您认为该思维片段多大程度上是关注【未来】的事情。[1:完全没有; 9:几乎都是]",
    "25.您认为该思维片段多大程度上是关注【当下】的事情。[1:完全没有; 9:几乎都是]",
    "26.您认为该思维片段反映说话者【反刍】的程度。（反刍：对负性情绪本身及其原因和可能后果的反复思考。）[1:完全没有; 9:几乎都是]",
]


In [10]:
QUESTION_IDS = list(range(1, 27))

SYSTEM_PROMPT = """你是一个严格的心理学专家，请根据下面的问题，给出你对该思维片段的评分。
必须输出JSON对象，键为q1...q26，值为1-9整数。
禁止输出任何解释或额外文本。"""

Q_BLOCK = "\n".join([f"Q{i}: {q}" for i, q in zip(QUESTION_IDS, QUESTIONS)])

def _extract_json(raw):
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        raise ValueError("No JSON found")
    return json.loads(m.group(0))

def rate_26dims(text, retries=3):
    if not isinstance(text, str) or text.strip() == "":
        return {f"q{i}": None for i in QUESTION_IDS}

    user_prompt = f"""请按Q1-Q26评分：

{Q_BLOCK}

文本：{text}

只输出JSON，如：
{{"q1":4,"q2":3,...,"q26":5}}"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gemini-3.5-flash",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                stream=False,
                temperature=0,
                response_format={"type": "json_object"}
            )

            data = _extract_json(response.choices[0].message.content)

            return {f"q{i}": int(data.get(f"q{i}", None))
                    if str(data.get(f"q{i}", "")).isdigit()
                    else None
                    for i in QUESTION_IDS}

        except Exception as e:
            print("Retry:", e)
            time.sleep(2 ** attempt)

    return {f"q{i}": None for i in QUESTION_IDS}

In [11]:

file_path = "D:/2025THU/SELF_OTHER/LLMRating_Task/draft/gemini/Gemini_batch_scoredFF.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

In [12]:
scores_series = df["Response"].apply(rate_26dims)
score_df = pd.DataFrame(list(scores_series))
df_out = pd.concat([df, score_df], axis=1)
df_out.to_excel("gemini3.5_rest.xlsx", index=False)